<a href="https://colab.research.google.com/github/awildt01/Credit-Scoring/blob/main/notebooks/4_applying_the_PD_Model_for_decision_makingi_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scorecards
In der Praxis übertragen Banken Modelle häufig in sogenannte Scorecards. Diese dienen der strukturierten Aufbereitung der Ergebnisse und fördern die Nachvollziehbarkeit insbesondere für nicht-technische Anspruchsgruppen. Im weiteren Verlauf wird die Umwandlung des Modells in eine Scorecard sowie die darauf basierende Berechnung von Kreditpunkten systematisch erläutert.

## 1.Grundidee – Logistische Regression im Kreditmodell

Wir haben ein Kreditrisikomodell mit einer logistischen Regression.  
Das Modell sagt uns: Wie hoch ist die Wahrscheinlichkeit, dass jemand den Kredit zurückzahlt („gut") oder ausfällt („schlecht")?

**Formel:**

$$
P(\text{gut}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_nx_n)}}
$$

Alternativ kann sie auch so dargestellt werden:
$$
P(\text{gut}) = \frac{e^{(\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_nx_n)}}{1 + e^{(\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_nx_n)}}
$$



Das klingt kompliziert – aber es ist eigentlich nur:  
**Intercept** + **Summe der passenden Dummy-Koeffizienten** → in die Formel einsetzen → **Wahrscheinlichkeit** ausrechnen.

## 2.Schritt-für-Schritt mit einem Mini-Beispiel

Stell dir vor, wir haben ein sehr einfaches Modell mit nur 3 Variablen:

*   **Intercept (Basiswert) $\beta_0$**: -1
*   **Kreditrating C $\beta_1$**: +0.7
*   **Hausbesitz Hypothek $\beta_2$**: +0.1
*   **Bundesstaat Kalifornien $\beta_3$**: +0.06

Jetzt haben wir einen Kreditnehmer mit diesen Eigenschaften:

*   Kreditrating = C  ($x_1 = 1$)
*   Hausbesitz = Hypothek  ($x_2 = 1$)
*   Bundesstaat = Kalifornien  ($x_3 = 1$)



### Schritt 1: Koeffizienten summieren (Lineare Kombination)

$$
\beta_0 + \beta_1x_1 + \beta_2x_2 + \beta_3x_3 = -1 + (0.7 \times 1) + (0.1 \times 1) + (0.06 \times 1) = -0.14
$$

Dieser Wert $(-0.14)$ ist der sogenannte **Logit-Wert** (oder Log-Odds).



### Schritt 2: In die Formel einsetzen

**Exponent berechnen (Odds):**
$$
\text{Odds} = e^{-0.14} \approx 0.87
$$

**Wahrscheinlichkeit berechnen:**
$$
P(\text{gut}) = \frac{0.87}{1 + 0.87} \approx 0.465
$$

Das Modell sagt: **46,5 % Wahrscheinlichkeit**, dass dieser Kreditnehmer den Kredit zurückzahlt.



## 3.Warum so viele Summen?

Weil jede Dummy-Variable nur 0 oder 1 ist:

*   Wenn die Eigenschaft **zutrifft** → der entsprechende Koeffizient $\beta_i$ wird addiert.
*   Wenn **nicht** → der Term $\beta_i \times 0$ fällt weg.

**Beispiel:**  
Kreditrating B hat vielleicht den Koeffizienten **+0.3**.  
Wenn jemand **nicht** B ist → Dummy $= 0$ → wir addieren nichts.



## 4.Was passiert in der Praxis?

Banken rechnen das nicht jedes Mal manuell, sondern verwandeln das Modell in **Scorecards**:

*   Jede Eigenschaft (z. B. Kreditrating, Hausstatus, Einkommen) bekommt Punkte.
*   Die Punkte werden addiert.
*   Am Ende ergibt sich ein Score (z. B. 650 Punkte).

Ab einem bestimmten Score sagt die Bank: **„Kredit genehmigt"** oder **„abgelehnt"**.

Das ist einfacher als Wahrscheinlichkeiten, weil Punkte leichter zu vergleichen sind.


**Kurz gesagt:**

1.  Wir addieren die passenden Koeffizienten zum **Logit-Wert**.
2.  Rechnen den Exponenten → das sind die **Odds**.
3.  Machen daraus eine **Wahrscheinlichkeit**.
4.  In der Praxis wird das Ganze in eine **Scorecard** umgewandelt.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
import os

#from google.colab import data_table
#data_table.enable_dataframe_formatter()

from google.colab import data_table
data_table.disable_dataframe_formatter()

# unbegrenzte Zeilen und Spalten anzeigen
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# **Creating a Scorecard**

**1.Erstellen des DataFrame mit Referenzkategorien (df_ref_categories)**


Für die spätere Scorecard-Erstellung wollen wir jede Dummy-Kategorie der im Modell verwendeten Variablen im Scorecard-Tableau abbilden — auch jene Kategorien, die beim Modelltraining als Referenz weggelassen wurden. Da Referenzkategorien beim Fitten keine Koeffizienten haben, fügen wir sie jetzt manuell mit Koeffizient = 0 (und p_value = NaN) ein. So ergibt sich später ein vollständiges, konsistentes Layout mit allen Kategorien pro Original-Feature.

Nach der Schätzung unseres finalen Logit-Modells (Probability of Default Modell) haben wir die Ergebnisse in einer **summary_table** gespeichert.
Diese Tabelle enthält:



- **Feature name**: den Namen der Variable oder Kategorie, die im Modell berücksichtigt wird.

- **Coefficients:** den geschätzten Regressionskoeffizienten (Einfluss auf die Ausfallwahrscheinlichkeit).

     - Positive Werte → senken die Ausfallwahrscheinlichkeit (günstigeres Kreditrisiko).

     - Negative Werte → erhöhen die Ausfallwahrscheinlichkeit (höheres Kreditrisiko).

- **p_values:** Signifikanzwerte, die anzeigen, ob der geschätzte Effekt statistisch relevant ist.

Da für jede kategoriale Variable eine Referenzkategorie festgelegt wird (Baseline), erscheint diese nicht in der **summary_table**.
Beispielsweise wird bei der Variable Grade die Kategorie grade:G als Referenz genutzt – daher finden sich nur die Kategorien grade:A bis grade:F in der Tabelle.



Um auch die Referenzkategorien in unserer finalen Scorecard darzustellen, haben wir die Liste **ref_categories** definiert.
Mit dieser Liste haben wir ein separates DataFrame df_ref_categories erstellt, in dem:

- **Coefficients = 0** (da die Referenz als Vergleichswert dient),

- **p_values = NaN** (nicht berechnet, da Referenz).



Anschließend können wir die **summary_table und df_ref_categories zusammenführen**, sodass eine vollständige Übersicht entsteht, in der alle Kategorien (inkl. Referenzen) enthalten sind.
Diese vollständige Tabelle bildet die Grundlage für die spätere Scorecard-Erstellung.

In [ ]:
# Und hier in der folgenden Liste behalten wir die Variablennamen für die Referenzkategorien bei,
# nur für die Variablen, die wir in unserem endgültigen PD-Modell verwendet haben.
ref_categories = ['grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'addr_state:ND_NE_IA_NV_FL_HI_AL',
'verification_status:Verified',
'purpose:educ__sm_b__wedd__ren_en__mov__house',
'initial_list_status:f',
'term:60',
'emp_length:0',
'mths_since_issue_d:>84',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'delinq_2yrs:0',
'inq_last_6mths:>6',
#'open_acc:0',
#'pub_rec:0-2',
'total_acc:0-15',
#'acc_now_delinq:0',
#'total_rev_hi_lim:<=5K',
'annual_inc:<20K',
'dti:>35',
'mths_since_last_delinq:0-3',
'mths_since_last_record:0-2']

In [ ]:
# Google Drive mounten
drive.mount('/content/drive')

# Zielordner definieren
folder_path = '/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks'
os.makedirs(folder_path, exist_ok=True)  # Ordner erstellen, falls nicht vorhanden

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
summary_table = pd.read_csv('/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/summary_table.csv', index_col = 0,header=0)

In [ ]:
summary_table

,Feature name,Coefficients,p_values
0,Intercept,-0.233534,NaN
1,grade:A,0.922083,2.245612e-26
2,grade:B,0.685872,1.957918e-30
3,grade:C,0.488966,1.392443e-18
4,grade:D,0.324109,8.033190e-10
5,grade:E,0.189386,5.465670e-05
6,grade:F,0.056816,2.443885e-01
7,home_ownership:OWN,0.083058,1.569663e-05
8,home_ownership:MORTGAGE,0.116799,3.374111e-22
9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,1.999357e-01


In [ ]:
# Vollständige Dezimaldarstellung (kann bei sehr kleinen Werten zu 0.0000 führen)
summary_table['p_values'] = summary_table['p_values'].apply(
    lambda x: f'{x:.6f}' if not pd.isna(x) else 'NaN'
)
# Anzeige der Tabelle
summary_table

,Feature name,Coefficients,p_values
0,Intercept,-0.233534,NaN
1,grade:A,0.922083,0.000000
2,grade:B,0.685872,0.000000
3,grade:C,0.488966,0.000000
4,grade:D,0.324109,0.000000
5,grade:E,0.189386,0.000055
6,grade:F,0.056816,0.244388
7,home_ownership:OWN,0.083058,0.000016
8,home_ownership:MORTGAGE,0.116799,0.000000
9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936


In [ ]:
df_ref_categories = pd.DataFrame(ref_categories, columns = ['Feature name'])
# Wir erstellen einen neuen Datenrahmen mit einer Spalte. Seine Werte sind die Werte aus der Liste „reference_categories“.
# Wir nennen ihn „Featurename“.
df_ref_categories['Coefficients'] = 0
# Fügt eine Spalte Coefficients hinzu und füllt sie mit 0. Warum 0?
# Weil beim Modelltraining keine Koeffizienten für Referenzen geschätzt wurden
# — ihre impliziten Koeffizienten sind 0 (die Referenz ist die Vergleichsbasis).
df_ref_categories['p_values'] = np.nan
# → Fügt eine Spalte p_values mit NaN ein, weil für diese (künstlich hinzugefügten) Zeilen keine statistischen Tests bzw. p-Werte berechnet wurden.
df_ref_categories

,Feature name,Coefficients,p_values
0,grade:G,0,NaN
1,home_ownership:RENT_OTHER_NONE_ANY,0,NaN
2,addr_state:ND_NE_IA_NV_FL_HI_AL,0,NaN
3,verification_status:Verified,0,NaN
4,purpose:educ__sm_b__wedd__ren_en__mov__house,0,NaN
5,initial_list_status:f,0,NaN
6,term:60,0,NaN
7,emp_length:0,0,NaN
8,mths_since_issue_d:>84,0,NaN
9,int_rate:>20.281,0,NaN


In [ ]:
df_scorecard = pd.concat([summary_table, df_ref_categories])
# Concatenates zwei dataframes.
df_scorecard = df_scorecard.reset_index()
# We reset the index of a dataframe.
df_scorecard

,index,Feature name,Coefficients,p_values
0,0,Intercept,-0.233534,NaN
1,1,grade:A,0.922083,0.000000
2,2,grade:B,0.685872,0.000000
3,3,grade:C,0.488966,0.000000
4,4,grade:D,0.324109,0.000000
5,5,grade:E,0.189386,0.000055
6,6,grade:F,0.056816,0.244388
7,7,home_ownership:OWN,0.083058,0.000016
8,8,home_ownership:MORTGAGE,0.116799,0.000000
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936


In [ ]:
df_scorecard['Original feature name'] = df_scorecard['Feature name'].str.split(':').str[0]
# Wir erstellen eine neue Spalte mit dem Namen „Original Feature name“,
# die den Wert der Spalte „Feature name“ bis zum Spaltensymbol enthält.

In [ ]:
df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name
0,0,Intercept,-0.233534,NaN,Intercept
1,1,grade:A,0.922083,0.000000,grade
2,2,grade:B,0.685872,0.000000,grade
3,3,grade:C,0.488966,0.000000,grade
4,4,grade:D,0.324109,0.000000,grade
5,5,grade:E,0.189386,0.000055,grade
6,6,grade:F,0.056816,0.244388,grade
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state


## **1.Score**
- Wenn wir für jede Variable die schlechteste Ausprägung nehmen und diese summieren, erhalten wir das niedrigste mögliche Modell-Logit-Ergebnis.

- Dieses Ergebnis wird später auf die Score-Skala (z. B. 300–850) transformiert.

- So stellen wir sicher, dass die Kombination „schlechteste Kategorie bei allen Variablen“ tatsächlich den Minimalwert der Skala (hier 300) ergibt.

- Mit .max() bestimmen wir die beste Ausprägung pro Variable.

- Wenn wir diese summieren, erhalten wir den theoretisch höchsten Score, der dann auf 850 gemappt wird.

In [ ]:
min_score = 300
max_score = 850

In [ ]:
df_scorecard.groupby('Original feature name')['Coefficients'].min()
# Gruppiert die Daten nach den Werten der Spalte „Ursprünglicher Merkmalsname“.
# Aggregiert die Daten in der Spalte „Koeffizienten“ und berechnet deren Minimum.

,Coefficients
Original feature name,
Intercept,-0.233534
addr_state,-0.017217
annual_inc,-0.157414
delinq_2yrs,-0.117282
dti,-0.116832
emp_length,0.000000
grade,0.000000
home_ownership,0.000000
initial_list_status,0.000000


In [ ]:
min_sum_coef = df_scorecard.groupby('Original feature name')['Coefficients'].min().sum()
# Bis zur Methode „min()“ ist alles wie in der obigen Zeile.
# Anschließend aggregieren wir weiter und summieren alle Minimalwerte.
min_sum_coef

np.float64(-0.9028357194654884)

In [ ]:
df_scorecard.groupby('Original feature name')['Coefficients'].max()
# Gruppiert die Daten nach den Werten der Spalte „Ursprünglicher Merkmalsname“.
# Aggregiert die Daten in der Spalte „Koeffizienten“ und berechnet deren Maximum.

,Coefficients
Original feature name,
Intercept,-0.233534
addr_state,0.632358
annual_inc,0.460002
delinq_2yrs,0.000000
dti,0.214882
emp_length,0.118364
grade,0.922083
home_ownership,0.116799
initial_list_status,0.039322


In [ ]:
max_sum_coef = df_scorecard.groupby('Original feature name')['Coefficients'].max().sum()
# Bis zur Methode „min()“ ist alles wie in der Zeile oben.
# Anschließend aggregieren wir weiter und summieren alle Maximalwerte.
max_sum_coef

np.float64(5.407542809776595)

## **2.Lineare Transformation der Koeffizienten**

Damit die Modellkoeffizienten in einen Scorebereich (z. B. 300–850 Punkte) übersetzt werden können, wendet man eine lineare Transformation an.

Bedeutung der einzelnen Teile **Text fett markieren**

- df_scorecard['Coefficients']
→ Das sind die β-Werte aus dem logistischen Modell.
Beispiel: Für "grade:A" = 0.92.

- (max_score - min_score)
→ Das ist der Zielbereich der Scorecard.
Beispiel: 850 – 300 = 550 Punkte.

- (max_sum_coef - min_sum_coef)
→ Das ist der Spannungsbereich der Modellkoeffizienten.

- max_sum_coef: maximale mögliche Summe der β-Werte (wenn eine Person in allen "besten Kategorien" ist).

- min_sum_coef: minimale mögliche Summe der β-Werte (wenn eine Person in allen "schlechtesten Kategorien" ist).

Damit haben wir die „Spanne“ der Logit-Skala.

**Multiplikation:**

→ Wir skalieren die einzelnen β-Werte so, dass sie in das gewünschte Punktesystem passen.

**Interpretierbarkeit:**

- Statt mit Logits zu hantieren, bekommt man eine verständliche Punkteskala (wie bei Schufa oder FICO).

- Vergleichbarkeit: Jeder Koeffizient trägt eine bestimmte Anzahl an Punkten zum Gesamtscore bei.

**Praxisnutzen: Kreditgeber können sagen:**

- "Ein Score von 700 bedeutet mittleres Risiko."

- "Ein Kunde mit 850 ist super, einer mit 300 sehr riskant."

## 3.Lineare Skalierung (ohne Verschiebung)
-  Wir skalieren die Roh-Koeffizienten proportional auf den Punktbereich (300–850).
- Ergebnis: Jede Ausprägung einer Variable bekommt Punkte zugewiesen.

Beispiel:

  - Einkommen hoch → +80 Punkte
 - Schulden hoch → –50 Punkte
 - Damit können wir die Einflüsse der Variablen interpretieren.

In [ ]:
# Wir skalieren die Roh-Koeffizienten proportional auf den Punktbereich (300–850).
# Ergebnis: Jede Ausprägung einer Variable bekommt Punkte zugewiesen.
# Beispiel:
# Einkommen hoch → +80 Punkte
# Schulden hoch → –50 Punkte
# Damit können wir die Einflüsse der Variablen interpretieren.
df_scorecard['Score - Calculation'] = df_scorecard['Coefficients'] * (max_score - min_score) / (max_sum_coef - min_sum_coef)
# Wir multiplizieren den Wert der Spalte „Koeffizienten“ mit dem Verhältnis der Differenzen zwischen
# Maximalpunktzahl und Minimalpunktzahl sowie der maximalen Summe der Koeffizienten und der minimalen Summe der Koeffizienten.
df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name,Score - Calculation
0,0,Intercept,-0.233534,NaN,Intercept,-20.354359
1,1,grade:A,0.922083,0.000000,grade,80.366875
2,2,grade:B,0.685872,0.000000,grade,59.779210
3,3,grade:C,0.488966,0.000000,grade,42.617335
4,4,grade:D,0.324109,0.000000,grade,28.248711
5,5,grade:E,0.189386,0.000055,grade,16.506545
6,6,grade:F,0.056816,0.244388,grade,4.951947
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership,7.239135
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership,10.179993
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state,-1.500634


## 4.Normalisierung des Gesamtscores

Hier normalisieren wir den Wert so, dass der **Gesamtscore eines Kunden** immer zwischen  
\(\text{min\_score} = 300\) und \(\text{max\_score} = 850\) liegt.

Formel = klassische **Min-Max-Normierung**:

$$
Score = \frac{Wert - \text{Min}}{\text{Max} - \text{Min}} \times (850 - 300) + 300
$$

Dadurch stellen wir sicher:

- Der **schlechteste Kunde** hat genau **300 Punkte**.  
- Der **beste Kunde** hat genau **850 Punkte**.



In [ ]:
# Wir dividieren die Differenz zwischen dem Wert in der Spalte „Koeffizienten“ und der minimalen Summe der Koeffizienten durch
# die Differenz zwischen der maximalen und der minimalen Summe der Koeffizienten.
# Anschließend multiplizieren wir das Ergebnis mit der Differenz zwischen der maximalen und der minimalen Punktzahl.
# Anschließend addieren wir die minimale Punktzahl.
df_scorecard.loc[0, 'Score - Calculation'] = (
    (df_scorecard.loc[0, 'Coefficients'] - min_sum_coef)
    / (max_sum_coef - min_sum_coef)
) * (max_score - min_score) + min_score

df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name,Score - Calculation
0,0,Intercept,-0.233534,NaN,Intercept,358.335001
1,1,grade:A,0.922083,0.000000,grade,80.366875
2,2,grade:B,0.685872,0.000000,grade,59.779210
3,3,grade:C,0.488966,0.000000,grade,42.617335
4,4,grade:D,0.324109,0.000000,grade,28.248711
5,5,grade:E,0.189386,0.000055,grade,16.506545
6,6,grade:F,0.056816,0.244388,grade,4.951947
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership,7.239135
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership,10.179993
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state,-1.500634


In [ ]:
#df_scorecard['Score - Calculation'][0] = ((df_scorecard['Coefficients'][0] - min_sum_coef) / (max_sum_coef - min_sum_coef)) * (max_score - min_score) + min_score
# We divide the difference of the value of the 'Coefficients' column and the minimum sum of coefficients by
# the difference of the maximum sum of coefficients and the minimum sum of coefficients.
# Then, we multiply that by the difference between the maximum score and the minimum score.
# Then, we add minimum score.
#df_scorecard

In [ ]:
df_scorecard['Score - Preliminary'] = df_scorecard['Score - Calculation'].round()
# Wir runden die Werte der Spalte „Score – Berechnung“.
df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name,Score - Calculation,Score - Preliminary
0,0,Intercept,-0.233534,NaN,Intercept,358.335001,358.0
1,1,grade:A,0.922083,0.000000,grade,80.366875,80.0
2,2,grade:B,0.685872,0.000000,grade,59.779210,60.0
3,3,grade:C,0.488966,0.000000,grade,42.617335,43.0
4,4,grade:D,0.324109,0.000000,grade,28.248711,28.0
5,5,grade:E,0.189386,0.000055,grade,16.506545,17.0
6,6,grade:F,0.056816,0.244388,grade,4.951947,5.0
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership,7.239135,7.0
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership,10.179993,10.0
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state,-1.500634,-2.0


In [ ]:
min_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Preliminary'].min().sum()
# Gruppiert die Daten nach den Werten der Spalte „Ursprünglicher Merkmalsname“.
# Aggregiert die Daten in der Spalte „Koeffizienten“ und berechnet deren Minimum.
# Summiert alle Minimumwerte.
min_sum_score_prel

np.float64(301.0)

In [ ]:
max_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Preliminary'].max().sum()
# Gruppiert die Daten nach den Werten der Spalte „Ursprünglicher Merkmalsname“.
# Aggregiert die Daten in der Spalte „Koeffizienten“ und berechnet deren Maximum.
# Summiert alle Maximalwerte.
max_sum_score_prel

np.float64(848.0)

In [ ]:
# Von der Maximalpunktzahl einer ursprünglichen Variable muss ein Wert abgezogen werden. Welcher? Wir bewerten anhand der Unterschiede.

In [ ]:
df_scorecard['Difference'] = df_scorecard['Score - Preliminary'] - df_scorecard['Score - Calculation']
df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name,Score - Calculation,Score - Preliminary,Difference
0,0,Intercept,-0.233534,NaN,Intercept,358.335001,358.0,-0.335001
1,1,grade:A,0.922083,0.000000,grade,80.366875,80.0,-0.366875
2,2,grade:B,0.685872,0.000000,grade,59.779210,60.0,0.220790
3,3,grade:C,0.488966,0.000000,grade,42.617335,43.0,0.382665
4,4,grade:D,0.324109,0.000000,grade,28.248711,28.0,-0.248711
5,5,grade:E,0.189386,0.000055,grade,16.506545,17.0,0.493455
6,6,grade:F,0.056816,0.244388,grade,4.951947,5.0,0.048053
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership,7.239135,7.0,-0.239135
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership,10.179993,10.0,-0.179993
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state,-1.500634,-2.0,-0.499366


In [ ]:
df_scorecard['Score - Final'] = df_scorecard['Score - Preliminary']

df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name,Score - Calculation,Score - Preliminary,Difference,Score - Final
0,0,Intercept,-0.233534,NaN,Intercept,358.335001,358.0,-0.335001,358.0
1,1,grade:A,0.922083,0.000000,grade,80.366875,80.0,-0.366875,80.0
2,2,grade:B,0.685872,0.000000,grade,59.779210,60.0,0.220790,60.0
3,3,grade:C,0.488966,0.000000,grade,42.617335,43.0,0.382665,43.0
4,4,grade:D,0.324109,0.000000,grade,28.248711,28.0,-0.248711,28.0
5,5,grade:E,0.189386,0.000055,grade,16.506545,17.0,0.493455,17.0
6,6,grade:F,0.056816,0.244388,grade,4.951947,5.0,0.048053,5.0
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership,7.239135,7.0,-0.239135,7.0
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership,10.179993,10.0,-0.179993,10.0
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state,-1.500634,-2.0,-0.499366,-2.0


In [ ]:
min_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Final'].min().sum()
# Gruppiert die Daten nach den Werten der Spalte „Ursprünglicher Merkmalsname“.
# Aggregiert die Daten in der Spalte „Koeffizienten“ und berechnet deren Minimum.
# Summiert alle Minimumwerte.
min_sum_score_prel

np.float64(301.0)

In [ ]:
max_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Final'].max().sum()
# Gruppiert die Daten nach den Werten der Spalte „Ursprünglicher Merkmalsname“.
# Aggregiert die Daten in der Spalte „Koeffizienten“ und berechnet deren Maximum.
# Summiert alle Maximalwerte.
max_sum_score_prel

np.float64(848.0)

## **5.Finales Scorecard**

Wir müssen zweimal prüfen:

- Min-Score: darf nicht größer als 300 werden.
→ also nichts bei den minimalen Kategorien anfassen.

- Max-Score: muss von 847 auf 850.
→ also 3 Punkte hinzufügen, nicht abziehen.

Das heißt: wir suchen die Features, die den maximalen Score bilden, und dort erhöhen wir die Kategorie, die am stärksten durch Rundung „verloren“ hat.

In [ ]:
# 1. Differenz zwischen gerundetem und originalem Score
df_scorecard["diff_rounding"] = df_scorecard["Score - Preliminary"] - df_scorecard["Score - Calculation"]

# 2. Kategorien identifizieren, die beim Maximum beteiligt sind
max_features = df_scorecard.groupby("Original feature name")["Score - Final"].idxmax()
df_max_features = df_scorecard.loc[max_features]

# 3. Kandidaten für Korrektur → die, die am meisten nach unten gerundet wurden
idx_to_fix = df_max_features["diff_rounding"].idxmin()  # hier: minimalster Wert = größte negative Rundung

# 4. Dort 3 Punkte addieren, um 847 → 850 zu heben
df_scorecard.loc[idx_to_fix, "Score - Final"] += 2

# 5. Nochmal prüfen
min_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Final'].min().sum()
max_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Final'].max().sum()

print("Min:", min_sum_score_prel, "Max:", max_sum_score_prel)


Min: 301.0 Max: 850.0


In [ ]:
# 1. Kategorien für Minimum ermitteln
min_features = df_scorecard.groupby("Original feature name")["Score - Final"].idxmin()
df_min_features = df_scorecard.loc[min_features]

# 2. Kandidat mit größter positive Rundungsabweichung suchen
idx_to_fix_min = df_min_features["diff_rounding"].idxmax()

# 3. Dort 1 Punkt abziehen
df_scorecard.loc[idx_to_fix_min, "Score - Final"] -= 1

# 4. Nochmal prüfen
min_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Final'].min().sum()
max_sum_score_prel = df_scorecard.groupby('Original feature name')['Score - Final'].max().sum()

print("Min:", min_sum_score_prel, "Max:", max_sum_score_prel)


Min: 300.0 Max: 850.0


In [ ]:
# -Scorecard Final Adjustment (Min = 300, Max = 850) ----

# Rundungsdifferenz berechnen (Preliminary - Calculation)
df_scorecard["diff_rounding"] = df_scorecard["Score - Preliminary"] - df_scorecard["Score - Calculation"]

# Start: Score - Final = Score - Preliminary
df_scorecard["Score - Final"] = df_scorecard["Score - Preliminary"]

# 1. Maximalen Score prüfen
max_features = df_scorecard.groupby("Original feature name")["Score - Final"].idxmax()
df_max_features = df_scorecard.loc[max_features]

if df_max_features["Score - Final"].sum() > 850:
    # Index der am stärksten "nach oben gerundeten" Kategorie finden
    idx_to_fix_max = df_max_features["diff_rounding"].idxmax()
    # 1 Punkt abziehen
    df_scorecard.loc[idx_to_fix_max, "Score - Final"] -= 1

# 2. Minimalen Score prüfen
min_features = df_scorecard.groupby("Original feature name")["Score - Final"].idxmin()
df_min_features = df_scorecard.loc[min_features]

if df_min_features["Score - Final"].sum() > 300:
    # Index der am stärksten "nach oben gerundeten" Kategorie finden
    idx_to_fix_min = df_min_features["diff_rounding"].idxmax()
    # 1 Punkt abziehen
    df_scorecard.loc[idx_to_fix_min, "Score - Final"] -= 1

# 3. Kontrolle: Min/Max Summen neu berechnen
min_sum_score_prel = df_scorecard.groupby('Original feature name')["Score - Final"].min().sum()
max_sum_score_prel = df_scorecard.groupby('Original feature name')["Score - Final"].max().sum()

print("✅ Final Check - Min:", min_sum_score_prel, "Max:", max_sum_score_prel)


✅ Final Check - Min: 300.0 Max: 848.0


**Finales Scorecard**
- Ergebnis: Score - Final mit ganzzahligen Punkten für jede Dummy-Kategorie,  validiert so dass min = min_score und max = max_score.

In [ ]:
# Google Drive mounten
drive.mount('/content/drive')

# Zielordner definieren
folder_path = '/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks'
os.makedirs(folder_path, exist_ok=True)  # Ordner erstellen, falls nicht vorhanden

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
inputs_test_with_ref_cat = pd.read_csv('/content/drive/MyDrive/Lending-Club-Credit-Scoring/notebooks/inputs_test_with_ref_cat.csv', index_col = 0,header=0)

In [ ]:
inputs_test_with_ref_cat.head()

,grade:A,grade:B,grade:C,grade:D,grade:E,grade:F,grade:G,home_ownership:RENT_OTHER_NONE_ANY,home_ownership:OWN,home_ownership:MORTGAGE,addr_state:ND_NE_IA_NV_FL_HI_AL,addr_state:OK_LA_NC_NM_MO_VA_NJ,addr_state:MD_TN_AZ_PA_MI,addr_state:DE_AR,addr_state:UT_MN_OH_IN_GA_ RI_WA,addr_state:OR_KY_MA,addr_state:MT_MS_SD,addr_state:WI_IL_CT_AK,addr_state:CO_SC,addr_state:KS_NH_WV_VT_ID,addr_state:WY_DC_ME,verification_status:Not Verified,verification_status:Source Verified,verification_status:Verified,purpose:educ__sm_b__wedd__ren_en__mov__house,purpose:credit_card,purpose:debt_consolidation,purpose:oth__med__vacation,purpose:major_purch__car__home_impr,initial_list_status:f,initial_list_status:w,term:36,term:60,emp_length:0,emp_length:1,emp_length:2-4,emp_length:5-6,emp_length:7-9,emp_length:10,mths_since_issue_d:<38,mths_since_issue_d:38-39,mths_since_issue_d:40-41,mths_since_issue_d:42-48,mths_since_issue_d:49-52,mths_since_issue_d:53-64,mths_since_issue_d:65-84,mths_since_issue_d:>84,int_rate:<9.548,int_rate:9.548-12.025,int_rate:12.025-15.74,int_rate:15.74-20.281,int_rate:>20.281,mths_since_earliest_cr_line:<140,mths_since_earliest_cr_line:141-164,mths_since_earliest_cr_line:165-247,mths_since_earliest_cr_line:248-270,mths_since_earliest_cr_line:271-352,mths_since_earliest_cr_line:>352,delinq_2yrs:0,delinq_2yrs:1-3,delinq_2yrs:4-16,delinq_2yrs:>=17,inq_last_6mths:0,inq_last_6mths:1-2,inq_last_6mths:3-6,inq_last_6mths:>6,total_acc:0-15,total_acc:16-70,total_acc:71-90,total_acc:>90,annual_inc:<20K,annual_inc:20K-30K,annual_inc:30K-40K,annual_inc:40K-50K,annual_inc:50K-60K,annual_inc:60K-70K,annual_inc:70K-80K,annual_inc:80K-90K,annual_inc:90K-100K,annual_inc:100K-120K,annual_inc:120K-140K,annual_inc:>140K,dti:<=1.4,dti:1.4-3.5,dti:3.5-7.7,dti:7.7-10.5,dti:10.5-16.1,dti:16.1-20.3,dti:20.3-21.7,dti:21.7-22.4,dti:22.4-35,dti:>35,mths_since_last_delinq:Missing,mths_since_last_delinq:0-3,mths_since_last_delinq:4-30,mths_since_last_delinq:31-56,mths_since_last_delinq:>=57,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86
362514,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
288564,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
213591,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
263083,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
165001,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0


In [ ]:
df_scorecard

,index,Feature name,Coefficients,p_values,Original feature name,Score - Calculation,Score - Preliminary,Difference,Score - Final,diff_rounding
0,0,Intercept,-0.233534,NaN,Intercept,358.335001,358.0,-0.335001,358.0,-0.335001
1,1,grade:A,0.922083,0.000000,grade,80.366875,80.0,-0.366875,80.0,-0.366875
2,2,grade:B,0.685872,0.000000,grade,59.779210,60.0,0.220790,60.0,0.220790
3,3,grade:C,0.488966,0.000000,grade,42.617335,43.0,0.382665,43.0,0.382665
4,4,grade:D,0.324109,0.000000,grade,28.248711,28.0,-0.248711,28.0,-0.248711
5,5,grade:E,0.189386,0.000055,grade,16.506545,17.0,0.493455,17.0,0.493455
6,6,grade:F,0.056816,0.244388,grade,4.951947,5.0,0.048053,5.0,0.048053
7,7,home_ownership:OWN,0.083058,0.000016,home_ownership,7.239135,7.0,-0.239135,7.0,-0.239135
8,8,home_ownership:MORTGAGE,0.116799,0.000000,home_ownership,10.179993,10.0,-0.179993,10.0,-0.179993
9,9,addr_state:OK_LA_NC_NM_MO_VA_NJ,-0.017217,0.199936,addr_state,-1.500634,-2.0,-0.499366,-2.0,-0.499366


## Caclulating Credit Score

In [ ]:
inputs_test_with_ref_cat.head()

In [ ]:
df_scorecard

In [ ]:
inputs_test_with_ref_cat_w_intercept = inputs_test_with_ref_cat

In [ ]:
inputs_test_with_ref_cat_w_intercept.insert(0, 'Intercept', 1)
# Wir fügen eine Spalte mit dem Index 0 in den Datenrahmen ein, also am Anfang des Datenrahmens.
# Der Name dieser Spalte lautet „Intercept“ und ihre Werte sind 1.

In [ ]:
inputs_test_with_ref_cat_w_intercept.head()

In [ ]:
inputs_test_with_ref_cat_w_intercept = inputs_test_with_ref_cat_w_intercept[df_scorecard['Feature name'].values]
# Hier behalten wir aus dem Dataframe „inputs_test_with_ref_cat_w_intercept“ nur die Spalten mit Spaltennamen bei,
# die genau den Zeilenwerten der Spalte „Featurename“ aus dem Dataframe „df_scorecard“ entsprechen.

In [ ]:
inputs_test_with_ref_cat_w_intercept.head()

In [ ]:
scorecard_scores = df_scorecard['Score - Final']

In [ ]:
inputs_test_with_ref_cat_w_intercept.shape

In [ ]:
scorecard_scores.shape

In [ ]:
scorecard_scores = scorecard_scores.values.reshape(102, 1)

In [ ]:
scorecard_scores.shape

In [ ]:
y_scores = inputs_test_with_ref_cat_w_intercept.dot(scorecard_scores)
# Hier multiplizieren wir die Werte jeder Zeile des Datenrahmens mit den Werten jeder Spalte der Variable,
# die ein Argument der „dot“-Methode ist, und summieren sie. Es ist im Wesentlichen die Summe der Produkte

In [ ]:
y_scores.head()

In [ ]:
y_scores.tail()

##  From credit score to PD